# 复盘

In [4]:
# 1. 导入必要库
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import operator
import os

1. 注意导入库时的langgraph.graph，不是直接langgraph
2. dotenv中导入的时load_dotenv而不是loadenv

In [5]:
# 2. LangGraph 的节点函数签名约定：每个节点接收 state，
# 返回一个 partial update（部分更新），LangGraph 会把它 merge 回全局 state。
class ResearchState(TypedDict):
    topic : str
    research_notes: str
    draft : str
    review_feedback: str
    final_report : str
    revision_count : Annotated[int, operator.add]


In [7]:
# 3.定义一个llm
load_dotenv()
llm = ChatOpenAI(
    model=os.getenv("GLM_model_1"),
    base_url=os.getenv("GLM_base_url"),
    api_key=os.getenv("GLM_API_KEY"),
    temperature=0.9
)

1. temperature注意拼写

In [23]:
# 4.定义节点函数
def research_node(state: ResearchState) -> dict:
    response = llm.invoke(
        f"请对以下主题进行调研，列出5个要点:\n\n{state['topic']}")
    return {"research_notes": response.content}

def write_node(state: ResearchState) -> dict:
    prompt=f"""
    主题：{state['topic']}
    笔记：{state['research_notes']}
    {"上次审阅意见:" + state.get('review_feedback', '')
    if state.get('review_feedback')
    else ''}

    请根据以上内容撰写三百字左右的内容
    """
    response = llm.invoke(prompt)
    return {"draft": response.content, "revision_count": 1}

def review_node(state: ResearchState) -> dict:
    response = llm.invoke(
        f"审阅以下内容，如果质量过关则回复'APPROVED',否则给出具体修改意见:\n\n{state['draft']}")
    return {"review_feedback": response.content}

def finalize_node(state: ResearchState) -> dict:
    return {"final_report": state['draft']}



1. f"审阅以下内容，如果质量过关则回复'APPROVED',否则给出具体修改意见:\n\n{state['draft']}")*需要在同一行，格式要求*
2. state.get('review_feedback', '')而不是state.get(state['review_feedback'], '')


In [24]:
# 路由函数，决定审阅节点后的走向：finalize或者revise
def should_revise(state: ResearchState) -> dict:
    if "APPROVED" in state["review_feedback"]:
        return "finalize"
    elif state["revision_count"] >= 3:
        return "finalize"
    else:
        return "rivise"


In [25]:
# 构建工作流图
workflow = StateGraph(ResearchState)
# 添加节点
workflow.add_node("research", research_node)
workflow.add_node("write", write_node)
workflow.add_node("review", review_node)
workflow.add_node("finalize", finalize_node)
# 设置入口
workflow.set_entry_point("research")
# 添加边
workflow.add_edge("research", "write")
workflow.add_edge("write"   , "review")
# 条件边
workflow.add_conditional_edges(
    "review",
    should_revise,
    {
        "finalize": "finalize",
        "revise"  : "write"
    }
)
workflow.add_edge("finalize", END)

1. 设置入口时是.set_entry_point而不是add_entry_nod
2. 设置入口时使用节点签名"research"而不是节点函数名"research_node"
3. 添加条件边时是add_conditional_edges而不是add_conditional_edge
4. 注意research拼写

In [26]:
# 编译并运行
app = workflow.compile()
result = app.invoke(
    {"topic": "人工智能专业的应届生应聘AI开发应用工程师应具备的技术栈",
    "revision_count": 0}
)
print(result["final_report"])

## AI应届生应聘“AI开发应用工程师”技术栈概览

人工智能专业应届生应聘“AI开发应用工程师”，需构建“理论＋工程＋实践”三位一体的技术栈。基础层面，须精通Python编程，熟悉Linux环境与Git协作开发；理论层面，掌握线性代数、概率统计等数学基础及经典机器学习算法，深入理解Transformer等深度学习架构，熟练使用PyTorch框架。当前最热门的大模型方向是核心竞争力，包括Prompt工程、RAG系统搭建、LoRA微调，以及LangChain等Agent开发框架与向量数据库的应用。工程化能力则是“应用”岗位的关键区分度，涵盖FastAPI模型服务化、ONNX/TensorRT推理优化、Docker容器化部署及云平台使用。此外，端到端项目经验、Kaggle等竞赛经历和GitHub作品集能显著提升竞争力。建议求职前仔细研究目标公司JD：大厂偏重工程与算法深度，中小企业更看重大模型落地与全栈能力，应针对性准备。
